# 🧠 PyAssistant Analytics — Tutorial Completo

**Dashboard personal de productividad con IA**  
*Built by Iván Arias G. · Ingeniero Mecatrónico | Data, Automation & AI*

---

## 📚 ¿Qué aprenderás en este notebook?

Este notebook es un **tutorial interactivo** que te enseña:

1. 🏗️ Cómo se diseña una aplicación **full-stack con FastAPI**
2. 🗄️ Cómo modelar datos con **SQLAlchemy ORM**
3. ✅ Cómo validar datos con **Pydantic**
4. 🌐 Cómo crear **endpoints REST** profesionales
5. 🤖 Cómo integrar **IA generativa** (Google Gemini)
6. 📊 Cómo hacer **agregaciones y analytics** en SQL
7. 🎨 Cómo conectar todo con un **frontend moderno**
8. 🐳 Cómo hacer **deploy** con Docker y Railway

**Recomendación:** Lee cada celda, ejecuta el código, y modifica los ejemplos para experimentar.

---

## 🎯 Contexto del Proyecto

**PyAssistant Analytics** es una aplicación real que:
- Rastrea el tiempo dedicado a diferentes actividades
- Visualiza patrones de productividad
- Genera insights personalizados usando IA
- Tiene un chat conversacional sobre tus datos

**Stack técnico completo:**
- **Backend:** Python 3.11 + FastAPI + SQLAlchemy + Pydantic
- **IA:** Google Gemini (google-genai SDK)
- **Base de datos:** SQLite (sin config externa)
- **Frontend:** HTML5 + CSS3 + Vanilla JavaScript + Chart.js
- **Deploy:** Docker + Railway

---

## 1. ⚙️ Setup del Entorno

Lo primero es instalar las dependencias y verificar que todo funciona.

### 1.1 Instalar dependencias

Ejecutar en la terminal:

In [ ]:
pip install fastapi uvicorn sqlalchemy pydantic python-dotenv google-genai


### 1.2 Verificar instalación

In [ ]:
import sys
import fastapi
import sqlalchemy
import pydantic
from google import genai

print(f"Python:     {sys.version.split()[0]}")
print(f"FastAPI:    {fastapi.__version__}")
print(f"SQLAlchemy: {sqlalchemy.__version__}")
print(f"Pydantic:   {pydantic.__version__}")
print(f"google-genai: {genai.__version__}")


**¿Qué acabamos de hacer?** Verificamos que las librerías están instaladas correctamente. Las versiones mínimas son importantes porque algunas features (como `Annotated` en FastAPI) requieren versiones recientes.

## 2. 🏗️ Arquitectura del Proyecto

Antes de escribir código, entendamos **cómo se organizan las piezas**.

### 2.1 Patrón por capas

La aplicación sigue el patrón **MVC + Service Layer**:

```
┌─────────────────────────────────────────┐
│         Frontend (HTML/JS)              │  ← Capa de presentación
└──────────────┬──────────────────────────┘
               │ HTTP (JSON)
┌──────────────▼──────────────────────────┐
│         API Layer (FastAPI)             │  ← Recibe requests
│  - activities.py (CRUD)                 │
│  - analytics.py (agregaciones)          │
│  - chat.py (IA)                         │
└──────────────┬──────────────────────────┘
               │
┌──────────────▼──────────────────────────┐
│      Business Logic (Python)            │  ← Procesa datos
│  - Validación con Pydantic              │
│  - Cálculos de analytics                │
│  - Llamadas a Gemini                    │
└──────────────┬──────────────────────────┘
               │
┌──────────────▼──────────────────────────┐
│      Data Layer (SQLAlchemy)            │  ← Persiste datos
│  - Models = tablas SQL                  │
│  - ORM traduce Python → SQL             │
└──────────────┬──────────────────────────┘
               │
┌──────────────▼──────────────────────────┐
│         SQLite Database                 │  ← Almacenamiento
└─────────────────────────────────────────┘
```

**¿Por qué esta estructura?**
- **Separación de responsabilidades:** cada capa hace una cosa
- **Testeable:** puedes testear cada capa por separado
- **Escalable:** puedes cambiar SQLite por PostgreSQL sin tocar la lógica
- **Mantenible:** nuevos features se agregan en la capa correcta

### 2.2 Flujo de un Request

Veamos qué pasa cuando el frontend pide datos:

```
1. Frontend: fetch('/api/analytics/summary?days=30')
   ↓
2. FastAPI recibe el request, valida los parámetros
   ↓
3. Endpoint llama a la lógica de analytics
   ↓
4. SQLAlchemy ejecuta: SELECT SUM(duration), AVG(score) ...
   ↓
5. Procesa resultados en Python
   ↓
6. FastAPI serializa a JSON y envía response
   ↓
7. Frontend recibe JSON, renderiza con Chart.js
```

**Tiempo total:** ~10-50ms (depende de la query)

## 3. 🔧 Configuración

Empezamos con el archivo de configuración. Este patrón se llama **"12-factor app"**: toda config viene de variables de entorno, no del código.

### 3.1 config.py — Carga de variables de entorno

**Concepto clave:** `pydantic-settings` lee automáticamente las variables de entorno (o un archivo `.env`) y las valida con tipos.

In [ ]:
# config.py
from pydantic_settings import BaseSettings, SettingsConfigDict
from typing import List


class Settings(BaseSettings):
    # Pydantic-settings busca automáticamente en .env
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        case_sensitive=True,  # Las vars son case-sensitive
    )

    # Cada campo es una variable de entorno
    APP_NAME: str = "PyAssistant Analytics"
    DEBUG: bool = True
    DATABASE_URL: str = "sqlite:///./pyassistant.db"
    GEMINI_API_KEY: str = "your_gemini_api_key_here"
    ALLOWED_ORIGINS: str = "http://localhost:3000,http://localhost:8000"
    HOST: str = "0.0.0.0"
    PORT: int = 8000

    # Propiedad calculada: convierte string a lista
    @property
    def ALLOWED_ORIGINS_LIST(self) -> List[str]:
        return [origin.strip() for origin in self.ALLOWED_ORIGINS.split(",")]


# Singleton: una sola instancia para toda la app
settings = Settings()


**¿Cómo funciona?**

1. Cuando creas `Settings()`, pydantic-settings busca:
   - Primero: variables de entorno del sistema (`os.environ`)
   - Segundo: archivo `.env` en el directorio actual
2. Convierte cada variable al tipo declarado (`str`, `bool`, `int`)
3. Si falta una variable requerida, lanza error
4. Si hay valor por default, lo usa

**Ventaja:** puedes tener diferentes configs en dev/staging/production sin cambiar código.

### 3.2 Ejemplo práctico: leyendo config

In [ ]:
# Simulando lectura de .env
import os
os.environ['GEMINI_API_KEY'] = 'AIzaSyDemo_xxxxx'
os.environ['DEBUG'] = 'True'

# En la app real, se importaría así:
# from config import settings
# Pero aquí simulamos:

class FakeSettings:
    DEBUG: bool = True
    GEMINI_API_KEY: str = 'tu_key_aqui'
    PORT: int = 8000

settings = FakeSettings()

print(f"App: PyAssistant Analytics")
print(f"Debug mode: {settings.DEBUG}")
print(f"API Key configurada: {'✅' if settings.GEMINI_API_KEY != 'tu_key_aqui' else '❌'}")
print(f"Puerto: {settings.PORT}")


## 4. 🗄️ Modelos de Base de Datos (SQLAlchemy)

Los **modelos** son clases Python que representan tablas SQL. Es como escribir SQL pero con código Python.

### 4.1 Concepto de ORM

**ORM (Object-Relational Mapping)** traduce entre:
- **Objetos Python** (clases, instancias) ↔ **Filas SQL** (tablas, registros)

**Ejemplo:**
```python
# En vez de escribir SQL:
cursor.execute("INSERT INTO activities (title, duration) VALUES (?, ?)", ("Coding", 60))

# Escribes Python:
activity = Activity(title="Coding", duration_minutes=60)
db.add(activity)
db.commit()
```

**Ventajas:**
- Menos errores de sintaxis SQL
- Autocompletado en el editor
- Portabilidad (cambias de SQLite a PostgreSQL sin cambiar código)
- Relaciones entre tablas más fáciles

### 4.2 Modelo Category (más simple)

In [ ]:
# models/category.py
from sqlalchemy import Column, Integer, String, DateTime
from sqlalchemy.orm import relationship
from datetime import datetime
from database import Base


class Category(Base):
    __tablename__ = "activities_categories"  # Nombre de la tabla en SQL

    # Columnas de la tabla
    id = Column(Integer, primary_key=True, index=True)
    name = Column(String(100), nullable=False, unique=True, index=True)
    color = Column(String(7), nullable=False, default="#3B82F6")  # hex color
    icon = Column(String(50), nullable=True)  # emoji
    description = Column(String(300), nullable=True)
    created_at = Column(DateTime, default=datetime.utcnow)

    # Relación: una categoría tiene muchas actividades
    activities = relationship(
        "Activity",
        back_populates="category",
        cascade="all, delete-orphan"  # Si borras la categoría, se borran sus actividades
    )


**Conceptos clave:**

1. **`__tablename__`**: nombre de la tabla en la base de datos
2. **`Column(Integer, ...)`**: define una columna. Los tipos son de SQLAlchemy
3. **`primary_key=True`**: esta columna es la llave primaria
4. **`index=True`**: crea un índice SQL (acelera búsquedas por esta columna)
5. **`unique=True`**: no puede haber dos categorías con el mismo nombre
6. **`nullable=False`**: el campo es obligatorio (NOT NULL en SQL)
7. **`relationship(...)`**: define relación con otra tabla (Foreign Key)
8. **`cascade="all, delete-orphan"`**: si borras padre, se borran hijos

### 4.3 Modelo Activity (más complejo, con relación)

In [ ]:
# models/activity.py
from sqlalchemy import Column, Integer, String, DateTime, Float, ForeignKey, Text
from sqlalchemy.orm import relationship
from datetime import datetime
from database import Base


class Activity(Base):
    __tablename__ = "activities_activities"

    id = Column(Integer, primary_key=True, index=True)
    title = Column(String(200), nullable=False, index=True)
    description = Column(Text, nullable=True)  # Text largo, sin límite
    category_id = Column(Integer, ForeignKey("activities_categories.id"), nullable=False)
    duration_minutes = Column(Float, nullable=False)  # Float para permitir 1.5h
    start_time = Column(DateTime, nullable=False, default=datetime.utcnow, index=True)
    end_time = Column(DateTime, nullable=True)
    productivity_score = Column(Integer, nullable=True)  # 1-10
    tags = Column(String(500), nullable=True)  # comma-separated
    created_at = Column(DateTime, default=datetime.utcnow)
    updated_at = Column(DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)

    # Relación: cada actividad pertenece a UNA categoría
    category = relationship("Category", back_populates="activities")


**Diferencias con Category:**

- **`ForeignKey("categories.id")`**: crea la relación SQL. No puedes tener una actividad sin categoría
- **`Float`**: permite decimales (1.5 horas = 90 minutos)
- **`Text`**: para descripciones largas (vs `String(200)` que limita a 200 chars)
- **`onupdate=datetime.utcnow`**: actualiza automáticamente cada vez que modificas el registro
- **`back_populates`**: es la "otra parte" de la relación

### 4.4 ¿Cómo se ven en SQL?

Veamos qué SQL genera SQLAlchemy:

In [ ]:
# Este código NO modifica la DB, solo nos muestra el SQL
from sqlalchemy import create_engine
from sqlalchemy.schema import CreateTable
from database import Base
from models import Activity, Category

engine = create_engine("sqlite:///:memory:")

# Genera el SQL CREATE TABLE
print("=" * 60)
print("SQL para crear tabla 'categories':")
print("=" * 60)
print(str(CreateTable(Category.__table__).compile(engine)))

print()
print("=" * 60)
print("SQL para crear tabla 'activities':")
print("=" * 60)
print(str(CreateTable(Activity.__table__).compile(engine)))


## 5. ✅ Validación con Pydantic

Los **schemas** son como "contratos" que definen qué datos acepta tu API. Si alguien envía datos inválidos, Pydantic los rechaza automáticamente.

### 5.1 ¿Por qué validar?

Imagina este endpoint:
```
POST /api/activities/
Body: {"title": "", "duration_minutes": -100, "category_id": 999}
```

Sin validación:
- Se crearía una actividad con título vacío ❌
- Duración negativa ❌
- Categoría que no existe ❌

Con Pydantic:
- FastAPI rechaza el request antes de tocar la DB ✅
- Devuelve un error 422 con detalles ✅
- Tu DB nunca se corrompe ✅

### 5.2 Schema ActivityCreate (datos de entrada)

In [ ]:
# schemas/activity.py
from pydantic import BaseModel, Field, ConfigDict
from datetime import datetime
from typing import Optional


class ActivityBase(BaseModel):
    # Field() agrega validaciones y metadata
    title: str = Field(..., min_length=1, max_length=200)
    description: Optional[str] = Field(None)
    category_id: int = Field(..., gt=0)  # gt=0: greater than 0
    duration_minutes: float = Field(..., gt=0, le=1440)  # max 24h
    start_time: datetime = Field(...)
    end_time: Optional[datetime] = None
    productivity_score: Optional[int] = Field(None, ge=1, le=10)
    tags: Optional[str] = None


class ActivityCreate(ActivityBase):
    """Schema para crear actividades. Hereda todas las validaciones."""
    pass


class ActivityResponse(ActivityBase):
    """Schema para devolver actividades. Incluye campos de la DB."""
    id: int
    created_at: datetime
    updated_at: datetime
    category_name: Optional[str] = None
    category_color: Optional[str] = None

    # Permite crear desde un modelo SQLAlchemy
    model_config = ConfigDict(from_attributes=True)


**Validaciones automáticas que obtienes:**

| Validación | Ejemplo | Error |
|------------|---------|-------|
| `min_length=1` | `title=""` | "String should have at least 1 character" |
| `gt=0` | `duration_minutes=-5` | "Input should be greater than 0" |
| `le=1440` | `duration_minutes=2000` | "Input should be less than or equal to 1440" |
| `ge=1, le=10` | `productivity_score=15` | "Input should be less than or equal to 10" |
| Type check | `title=123` (int) | "Input should be a valid string" |

### 5.3 Ejemplo: probando validaciones

In [ ]:
from pydantic import ValidationError

# Simulando un schema simple
from pydantic import BaseModel, Field
from typing import Optional

class TestActivity(BaseModel):
    title: str = Field(..., min_length=1, max_length=200)
    duration_minutes: float = Field(..., gt=0, le=1440)
    productivity_score: Optional[int] = Field(None, ge=1, le=10)


# Caso 1: Datos válidos
try:
    activity = TestActivity(title="Coding session", duration_minutes=60, productivity_score=8)
    print(f"✅ Válido: {activity}")
except ValidationError as e:
    print(f"❌ Error: {e}")

# Caso 2: Título vacío
try:
    activity = TestActivity(title="", duration_minutes=60)
except ValidationError as e:
    print(f"\n❌ Título vacío: {e.errors()[0]['msg']}")

# Caso 3: Duración negativa
try:
    activity = TestActivity(title="Test", duration_minutes=-10)
except ValidationError as e:
    print(f"❌ Duración negativa: {e.errors()[0]['msg']}")

# Caso 4: Duración > 24h
try:
    activity = TestActivity(title="Test", duration_minutes=3000)
except ValidationError as e:
    print(f"❌ Duración > 24h: {e.errors()[0]['msg']}")

# Caso 5: Score fuera de rango
try:
    activity = TestActivity(title="Test", duration_minutes=60, productivity_score=15)
except ValidationError as e:
    print(f"❌ Score fuera de rango: {e.errors()[0]['msg']}")


## 6. 📝 Operaciones CRUD

CRUD = **C**reate, **R**ead, **U**pdate, **D**elete. Son las 4 operaciones básicas de cualquier sistema.

### 6.1 Anatomía de un endpoint FastAPI

Vamos a desglosar línea por línea un endpoint real:

In [ ]:
# api/activities.py
from fastapi import APIRouter, Depends, HTTPException
from sqlalchemy.orm import Session
from typing import List
from database import get_db
from models import Activity, Category
from schemas import ActivityCreate, ActivityResponse

router = APIRouter()


@router.get("/", response_model=List[ActivityResponse])
async def list_activities(
    skip: int = 0,
    limit: int = 100,
    category_id: int = None,
    db: Session = Depends(get_db),  # ← Inyección de dependencias
):
    """Lista actividades con filtros opcionales."""
    query = db.query(Activity)

    # Filtros dinámicos
    if category_id:
        query = query.filter(Activity.category_id == category_id)

    # Paginación + orden
    activities = query.order_by(Activity.start_time.desc()).offset(skip).limit(limit).all()

    return activities


**Desglose línea por línea:**

```python
@router.get("/", response_model=List[ActivityResponse])
```
- `@router.get("/")` → Define un endpoint GET en la ruta `/`
- `response_model=...` → FastAPI validará la respuesta con este schema

```python
async def list_activities(skip: int = 0, limit: int = 100, ...):
```
- `async def` → Función asíncrona (puede manejar miles de requests simultáneas)
- Parámetros con tipos → FastAPI sabe qué validar
- `= 0`, `= 100` → Valores por defecto

```python
db: Session = Depends(get_db)
```
- `Depends()` → Inyección de dependencias. FastAPI llama a `get_db()` automáticamente
- Pasa una sesión de DB a tu función
- Al terminar, FastAPI cierra la sesión automáticamente

```python
query = db.query(Activity)
```
- Crea un query de SQLAlchemy
- Todavía NO se ejecuta (lazy)

```python
query = query.filter(Activity.category_id == category_id)
```
- Agrega un WHERE clause
- Se puede llamar varias veces para múltiples filtros

```python
query.order_by(Activity.start_time.desc()).offset(skip).limit(limit).all()
```
- `.order_by()` → ORDER BY
- `.offset(skip).limit(limit)` → LIMIT/OFFSET para paginación
- `.all()` → AHORA sí ejecuta el query y devuelve una lista

### 6.2 Ejemplo: Probando el query builder de SQLAlchemy

In [ ]:
# Simulemos el query building
from sqlalchemy.orm import Query

# Query vacío
q1 = query.filter(Activity.category_id == 1)
print("Query con filtro:", str(q1)[:100], "...")

# Múltiples filtros (se encadenan con AND)
q2 = q1.filter(Activity.duration_minutes > 30)
print("\nQuery con 2 filtros:", str(q2)[:120], "...")

# Ordenamiento
q3 = q2.order_by(Activity.start_time.desc())
print("\nCon ORDER BY:", str(q3)[:150], "...")


## 7. 📊 Analytics: Agregaciones y Estadísticas

Esta es la parte más interesante: convertir datos crudos en insights.

### 7.1 Endpoint: Resumen General

In [ ]:
# api/analytics.py
from fastapi import APIRouter, Depends, Query
from sqlalchemy.orm import Session
from sqlalchemy import func
from datetime import datetime, timedelta
from database import get_db
from models import Activity, Category

router = APIRouter()


@router.get("/summary")
async def get_summary(days: int = Query(30, ge=1, le=365), db: Session = Depends(get_db)):
    """
    Calcula estadísticas agregadas de los últimos N días.

    Devuelve:
    - total_activities: número de actividades
    - total_hours: horas totales trabajadas
    - average_productivity: score promedio (1-10)
    - most_productive_day: día con más horas
    """
    # Calcula fecha de inicio
    start_date = datetime.utcnow() - timedelta(days=days)

    # Trae TODAS las actividades del período
    # (En producción, harías agregaciones en SQL, no en Python)
    activities = db.query(Activity).filter(Activity.start_time >= start_date).all()

    if not activities:
        return {"message": "No hay datos en este período"}

    # Agregación 1: total de horas
    total_minutes = sum(a.duration_minutes for a in activities)

    # Agregación 2: productividad promedio
    scores = [a.productivity_score for a in activities if a.productivity_score]
    avg_score = sum(scores) / len(scores) if scores else 0

    # Agregación 3: agrupar por día para encontrar el más productivo
    daily_hours = {}
    for a in activities:
        day = a.start_time.strftime("%Y-%m-%d")
        daily_hours[day] = daily_hours.get(day, 0) + a.duration_minutes / 60

    # Encuentra el día con más horas
    most_productive_day = max(daily_hours.items(), key=lambda x: x[1])

    return {
        "period_days": days,
        "total_activities": len(activities),
        "total_hours": round(total_minutes / 60, 2),
        "average_productivity": round(avg_score, 2),
        "most_productive_day": {
            "date": most_productive_day[0],
            "hours": round(most_productive_day[1], 2),
        },
    }


**Desglose:**

1. **Filtro temporal:** `Activity.start_time >= start_date`
2. **Carga en memoria:** `.all()` trae todo a Python (ok para <10K registros)
3. **Agregaciones en Python:** `sum()`, `max()`, comprehensions
4. **Grouping:** usamos un dict para agrupar por día
5. **`max(..., key=lambda x: x[1])`:** encuentra el día con más horas (mayor valor)

### 7.2 Ejemplo: las agregaciones en acción

In [ ]:
# Simulando datos
actividades_fake = [
    {"fecha": "2026-01-01", "horas": 6, "score": 8},
    {"fecha": "2026-01-02", "horas": 8, "score": 9},
    {"fecha": "2026-01-03", "horas": 4, "score": 6},
    {"fecha": "2026-01-04", "horas": 9, "score": 10},
    {"fecha": "2026-01-05", "horas": 7, "score": 8},
]

# Total
total_horas = sum(a["horas"] for a in actividades_fake)
print(f"Total horas: {total_horas}")

# Promedio
scores = [a["score"] for a in actividades_fake]
promedio = sum(scores) / len(scores)
print(f"Promedio score: {promedio:.2f}")

# Mejor día
mejor = max(actividades_fake, key=lambda a: a["horas"])
print(f"Mejor día: {mejor['fecha']} con {mejor['horas']}h")


### 7.3 Endpoint: Distribución por Categoría (usando GROUP BY de SQL)

In [ ]:
@router.get("/by-category")
async def get_by_category(days: int = Query(30, ge=1, le=365), db: Session = Depends(get_db)):
    start_date = datetime.utcnow() - timedelta(days=days)

    # Esta vez usamos SQL para hacer la agregación (más eficiente)
    results = (
        db.query(
            Category.id,
            Category.name,
            Category.color,
            Category.icon,
            func.count(Activity.id).label("count"),  # COUNT(*)
            func.sum(Activity.duration_minutes).label("total_minutes"),  # SUM()
            func.avg(Activity.productivity_score).label("avg_score"),  # AVG()
        )
        .join(Activity, Activity.category_id == Category.id)  # JOIN
        .filter(Activity.start_time >= start_date)  # WHERE
        .group_by(Category.id)  # GROUP BY
        .all()
    )

    return [
        {
            "category_id": r.id,
            "name": r.name,
            "color": r.color,
            "icon": r.icon,
            "activity_count": r.count,
            "total_hours": round(r.total_minutes / 60, 2) if r.total_minutes else 0,
            "average_productivity": round(r.avg_score, 2) if r.avg_score else 0,
        }
        for r in results
    ]


**SQL generado:**
```sql
SELECT
    categories.id, categories.name, categories.color, categories.icon,
    COUNT(activities.id) AS count,
    SUM(activities.duration_minutes) AS total_minutes,
    AVG(activities.productivity_score) AS avg_score
FROM categories
JOIN activities ON activities.category_id = categories.id
WHERE activities.start_time >= ?
GROUP BY categories.id
```

**¿Por qué GROUP BY en SQL y no en Python?**
- Más eficiente: la DB procesa millones de filas
- Menos transferencia de datos
- Aprovecha índices

## 8. 🤖 Integración con IA (Google Gemini)

La pieza más "mágica": usar un LLM para generar análisis en lenguaje natural.

### 8.1 Configuración del Cliente

In [ ]:
# ai/gemini.py
from google import genai
from config import settings
import logging

logger = logging.getLogger(__name__)

# Solo configuramos Gemini si hay API key real
if settings.GEMINI_API_KEY and settings.GEMINI_API_KEY != "your_gemini_api_key_here":
    try:
        # Nuevo SDK (google-genai, no el deprecado google-generativeai)
        client = genai.Client(api_key=settings.GEMINI_API_KEY)
        MODEL_ID = "gemini-2.0-flash-exp"  # Modelo rápido y económico
        AI_ENABLED = True
        logger.info("✅ Gemini AI configured successfully")
    except Exception as e:
        logger.error(f"❌ Failed to configure Gemini: {e}")
        AI_ENABLED = False
else:
    AI_ENABLED = False
    logger.warning("⚠️ GEMINI_API_KEY not configured - AI features disabled")


**Conceptos clave:**

1. **Lazy initialization:** Solo configuramos Gemini si hay API key
2. **Modelo `gemini-2.0-flash-exp`:** Versión rápida y barata (~$0.0001 por request)
3. **Try/except:** Si falla la config, la app sigue funcionando sin IA
4. **Fallback graceful:** Sin IA, el dashboard funciona, solo se deshabilitan insights

### 8.2 Función de Insights (Prompt Engineering)

In [ ]:
async def generate_insights(activities_data: dict) -> str:
    """
    Genera insights personalizados sobre datos de productividad.
    """
    if not AI_ENABLED:
        return "⚠️ AI features are disabled. Configure GEMINI_API_KEY."

    # El SECRETO está en el prompt: claro, específico, con formato
    prompt = f"""You are a productivity analyst AI. Analyze the following activity tracking data and provide actionable insights in Spanish.

Data:
{activities_data}

Provide a concise analysis (max 200 words) covering:
1. 📊 Main patterns observed
2. ✅ What's working well
3. ⚠️ Areas for improvement
4. 💡 2-3 specific recommendations

Be direct, use emojis for readability, and focus on actionable advice."""

    try:
        # Llamada async al LLM
        response = await client.aio.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
        )
        return response.text
    except Exception as e:
        return f"❌ Error: {str(e)}"


**Anatomía de un buen prompt para LLMs:**

1. **Rol claro:** "You are a productivity analyst AI"
2. **Datos estructurados:** pasamos JSON/dict con la info
3. **Formato de salida:** especificamos qué queremos (4 secciones, emojis, max 200 palabras)
4. **Tono:** "Be direct, use emojis"
5. **Idioma:** "in Spanish" (importante para respuestas consistentes)

### 8.3 Función de Chat (RAG-like)

In [ ]:
async def answer_question(question: str, activities_data: dict) -> str:
    """
    Responde preguntas en lenguaje natural sobre los datos del usuario.

    Esto es similar a RAG (Retrieval-Augmented Generation):
    - "Retrieval" = cargamos los datos relevantes
    - "Augmented" = los añadimos al prompt
    - "Generation" = Gemini genera una respuesta contextual
    """
    if not AI_ENABLED:
        return "⚠️ AI features are disabled."

    prompt = f"""You are a personal productivity assistant. Answer the following question about the user's activity data.

User's question: "{question}"

Activity data context:
{activities_data}

Provide a clear, direct answer in the SAME language as the question. Use specific numbers from the data when possible. Be conversational but concise (max 150 words)."""

    try:
        response = await client.aio.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
        )
        return response.text
    except Exception as e:
        return f"❌ Error: {str(e)}"


**¿Por qué funciona?**

Cuando el usuario pregunta "¿Cuál fue mi mejor semana?", el LLM:
1. Recibe el contexto con datos agregados por semana
2. Busca en su conocimiento + los datos proporcionados
3. Genera una respuesta específica para ese usuario

**Limitaciones:**
- El LLM NO tiene acceso directo a la DB (no ejecuta SQL)
- Por eso le pasamos datos pre-agregados
- Para análisis más complejos, necesitaríamos function calling

## 9. 🎨 Frontend (HTML + JavaScript)

El frontend es deliberadamente simple: **HTML estático + JavaScript vanilla**. Sin React, Vue, ni build step.

### 9.1 ¿Por qué no usar React/Vue?

**Pros de vanilla JS:**
- ✅ Cero build step (más simple)
- ✅ Cero dependencias npm
- ✅ Más rápido de cargar
- ✅ Perfecto para dashboards pequeños

**Contras:**
- ❌ Más verboso para apps grandes
- ❌ No hay hot reload automático
- ❌ Manejo manual del DOM

**Para este proyecto:** vanilla JS es la opción correcta porque solo tenemos 5 vistas y datos simples.

### 9.2 Patrón del Frontend

In [ ]:
// frontend/js/app.js (extracto)

const API_BASE = window.location.origin + '/api';

// 1. Función helper para llamar la API
async function apiGet(endpoint) {
    const response = await fetch(`${API_BASE}${endpoint}`);
    return await response.json();
}

// 2. Navegación entre vistas (sin SPA router)
document.querySelectorAll('.nav-btn').forEach(btn => {
    btn.addEventListener('click', () => {
        const view = btn.dataset.view;

        // Oculta todas las vistas
        document.querySelectorAll('.view').forEach(v => v.classList.remove('active'));
        // Muestra solo la seleccionada
        document.getElementById(view).classList.add('active');

        // Carga datos específicos de la vista
        if (view === 'overview') loadOverview();
        else if (view === 'daily') loadDailyChart();
    });
});

// 3. Cargar resumen general
async function loadOverview() {
    const data = await apiGet('/analytics/summary?days=30');

    // Actualizar el DOM con los datos
    document.getElementById('totalActivities').textContent = data.total_activities;
    document.getElementById('totalHours').textContent = data.total_hours + 'h';
    document.getElementById('avgScore').textContent = data.average_productivity + '/10';
}


**Desglose:**

1. **`apiGet()`**: wrapper sobre `fetch()` para mantener consistencia
2. **Navegación manual:** sin router, solo toggleo de clases CSS
3. **Lazy loading:** cada vista carga sus datos cuando se muestra
4. **Actualización directa del DOM:** `el.textContent = ...`

### 9.3 Chart.js: Visualizaciones

In [ ]:
// Cargar gráfico de tendencia diaria
async function loadDailyChart() {
    const data = await apiGet('/analytics/daily?days=7');

    new Chart(document.getElementById('dailyChart'), {
        type: 'bar',
        data: {
            labels: data.map(d => d.date),
            datasets: [{
                label: 'Horas trabajadas',
                data: data.map(d => d.hours),
                backgroundColor: '#10366D',
            }, {
                label: 'Productividad (x10)',
                data: data.map(d => d.avg_score * 10),
                backgroundColor: '#2C8C57',
                yAxisID: 'y1',
            }],
        },
        options: {
            scales: {
                y: { beginAtZero: true, title: { text: 'Horas' } },
                y1: { position: 'right', title: { text: 'Score' } },
            },
        },
    });
}


**Conceptos de Chart.js:**

- **`type: 'bar'`** → Tipo de gráfico
- **`labels`** → Etiquetas del eje X
- **`datasets`** → Array de series de datos
- **`backgroundColor`** → Color de las barras
- **`yAxisID: 'y1'`** → Segundo eje Y (para comparar dos métricas)
- **`options.scales`** → Configuración de ejes

## 10. 🚢 Deploy con Docker y Railway

### 10.1 Dockerfile

In [ ]:
# Dockerfile
FROM python:3.11-slim

WORKDIR /app

# Instala dependencias del sistema
RUN apt-get update && apt-get install -y gcc

# Copia e instala dependencias Python
COPY backend/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copia el código
COPY backend/ .

# Expone el puerto
EXPOSE 8000

# Comando de inicio (producción)
CMD ["gunicorn", "main:app", "--workers", "2",
     "--worker-class", "uvicorn.workers.UvicornWorker",
     "--bind", "0.0.0.0:8000"]


**Conceptos:**

- **`FROM python:3.11-slim`**: imagen base ligera
- **`WORKDIR /app`**: directorio de trabajo dentro del contenedor
- **`pip install --no-cache-dir`**: no guarda cache (imagen más pequeña)
- **`gunicorn`**: servidor WSGI de producción (vs `uvicorn` que es para desarrollo)
- **`--workers 2`**: 2 procesos paralelos para manejar más requests

### 10.2 ¿Por qué Docker?

**Problema sin Docker:**
- "En mi máquina funciona" (pero no en producción)
- Diferentes versiones de Python en dev vs server
- Dependencias del sistema operativo diferentes

**Solución con Docker:**
- La app corre en un **contenedor aislado**
- Mismo ambiente en dev, staging, producción
- Deploy con un solo comando

## 11. 💪 Ejercicios Prácticos

Para consolidar lo aprendido, intenta estos ejercicios:

### Ejercicio 1: Agregar un nuevo campo

**Objetivo:** Agregar un campo `priority` (low, medium, high) a las actividades.

**Pasos:**
1. Modifica `models/activity.py` → agrega columna `priority`
2. Modifica `schemas/activity.py` → agrega el campo con validación
3. Crea un script de migración para actualizar la DB
4. Prueba con un POST

**Pista:** SQLAlchemy no modifica tablas existentes automáticamente. Necesitas Alembic o drop+recreate.

### Ejercicio 2: Nuevo endpoint de analytics

**Objetivo:** Crear `GET /api/analytics/streak` que devuelva la racha de días consecutivos con actividades.

**Lógica:**
1. Trae todas las fechas únicas con actividades
2. Ordena ascendente
3. Cuenta la racha más larga de días consecutivos
4. Devuelve también la racha actual (días desde hoy hacia atrás)

**Pista:** Usa `set()` para las fechas únicas, luego itera comparando fechas consecutivas.

### Ejercicio 3: Mejorar el prompt de IA

**Objetivo:** Experimenta con diferentes prompts para los insights.

**Pruebas a hacer:**
- Prompt más corto vs más largo
- Pedir formato JSON vs texto libre
- Especificar audiencia (manager vs tú mismo)
- Incluir comparaciones con períodos anteriores

**Mide:**
- ¿Cuál genera insights más accionables?
- ¿Cuál es más rápido?
- ¿Cuál respeta mejor el formato pedido?

### Ejercicio 4: Deploy y compartir

**Objetivo:** Hacer deploy real en Railway y compartir la URL.

**Pasos:**
1. Crea cuenta en github.com
2. Sube el código (sin .env ni .db)
3. Conecta con Railway
4. Configura GEMINI_API_KEY en variables de entorno
5. Obtén tu URL pública
6. Compártela en LinkedIn con un post como el sugerido en la guía

## 12. 📚 Resumen de Conceptos

Has aprendido:

| Concepto | Qué es | Por qué importa |
|----------|--------|-----------------|
| **FastAPI** | Framework web async para Python | Más rápido que Flask, genera docs auto |
| **Pydantic** | Validación de datos con tipos | Previene bugs antes de tocar la DB |
| **SQLAlchemy ORM** | Mapeo objeto-relacional | Escribes Python, no SQL directo |
| **REST API** | Patrón de arquitectura web | Estándar de la industria |
| **Async/Await** | Programación asíncrona | Maneja miles de conexiones |
| **Docker** | Contenedores | "Funciona en mi máquina" solved |
| **Prompt Engineering** | Arte de hablar con LLMs | Calidad del output depende del input |
| **Gemini API** | LLM de Google | Alternativa a OpenAI con tier gratis |

## 🎓 Siguientes Pasos

Ahora que entiendes el proyecto completo:

1. **Modifícalo:** agrega features propios (gamification, goals, export a PDF)
2. **Aprende más:** cada librería tiene docs oficiales excelentes
3. **Muestra tu trabajo:** publica en GitHub, LinkedIn, y tu CV
4. **Conecta con la comunidad:** r/Python, r/FastAPI, Discord de AI

**Recuerda:** El mejor código no es el más complejo, es el que otros pueden entender y mantener.

## 📬 Contacto

¿Dudas, sugerencias, o quieres mostrarme tu versión del proyecto?

- 📧 ivan.ariasg@hotmail.com
- 🔗 linkedin.com/in/ivan-arias-993587233
- 💻 github.com/IvanArias77

---

**¡Éxito en tu viaje de aprendizaje, Iván! 🚀**

*— Generado el 2026-07-08 · PyAssistant Analytics Tutorial*